# Image Processing Project - NYC Detective
Carlos Ponce (`cmp279`)  
Zachary Hunt (`zh362`)  
Mykyta Turpitka (`mt689`)

# Imports

In [ ]:
import os
import keras
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from skimage import io
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Dense, Flatten, Dropout
from keras.preprocessing.image import img_to_array, load_img
from tensorflow.keras.optimizers import SGD

## A - Data Loading

In [ ]:
input_directory = "./data/"
photo_directory = input_directory + "processed/"
with open(input_directory + "PhotoTable.p", 'rb') as pickle_file:
    photo_info = pickle.load(pickle_file)
photo_info

In [ ]:
def prepare_data(list_of_images):
    x = [] # images as arrays
    for image in list_of_images:
        x.append(img_to_array(load_img(image,target_size=sample_photo.shape)))
    return np.array(x)

In [ ]:
train = photo_directory
training = [train+i for i in os.listdir(train)]
sample_photo = io.imread(training[0])
image_shape = (124, 187, 1)
train_X_raw = prepare_data(training)
train_Y_raw = photo_info["Target"]

print("Train shapes:", train_X_raw.shape, "->", train_Y_raw.shape)

In [ ]:
plt.imshow(train_X_raw[0, :, :, 0])

## B - Preprocessing

In [ ]:
# Reshape to single channel, scale down to [0, 1]
train_X = train_X_raw[:, :, :, :1] / 255.0
train_Y = train_Y_raw

## C - CNN Implementation

In [ ]:
def create_cnn(dropout=False, conv_layer2=False, learning_rate=0.01):
    # Define using Sequential
    model = Sequential()
    # Convolution Layer
    model.add(
        Conv2D(
            32,
            (3, 3),
            activation="relu",
            kernel_initializer="he_uniform",
            input_shape=image_shape,
        )
    )
    # Maxpooling Layer
    model.add(MaxPooling2D((2, 2)))
    
    if conv_layer2:
        # Convolution Layer
        model.add(
            Conv2D(
                64,
                (3, 3),
                activation="relu",
                kernel_initializer="he_uniform",
                input_shape=image_shape,
            )
        )
        # Maxpooling Layer
        model.add(MaxPooling2D((2, 2)))
    
    # Flatten Output
    model.add(Flatten())
    # Droupout Layer
    # if dropout:
    #     model.add(Dropout(0.5))
    # Dense Layer of 100 neurons
    model.add(Dense(100, activation="relu", kernel_initializer="he_uniform"))
    # Initialize Bias
    model.add(Dense(1, activation="linear"))  # Potentials: Sigmoid, linear
    # Initialize Optimizer
    opt = SGD(learning_rate=learning_rate, momentum=0.9)
    
    # Compile Model
    model.compile(loss="binary_crossentropy", optimizer=opt, metrics=["accuracy"])

    return model

## D - Training and Evaluating CNN

In [ ]:
test_set = np.random.choice(list(range(train_X.shape[0])), size=int(train_X.shape[0] * 0.1), replace=False)
train_set = [i for i in list(range(flat_X.shape[0])) if i not in test_set]

In [ ]:
model = create_cnn(conv_layer2=True, learning_rate=0.01)  # dropout=True,
epochs = model.fit(train_X[train_set], train_Y[train_set].astype(int).values, batch_size=32, epochs=125, validation_split=0.1, verbose=0)
epochs.history["accuracy"][-1], epochs.history["loss"][-1], epochs.history["val_accuracy"][-1]

### Classification

In [ ]:
new_york_middle = model.predict(train_X[train_set][train_Y[train_set]]).mean()
new_york_middle

In [ ]:
non_middle = model.predict(train_X[train_set][~train_Y[train_set]]).mean()
non_middle

In [ ]:
new_york_bias = 3 / 3  # Fraction between 0 and 1

In [ ]:
division = abs(new_york_middle - non_middle) * new_york_bias + min(new_york_middle, non_middle)

In [ ]:
predictions = (model.predict(train_X[test_set]) > division).astype(int).reshape(-1)
predictions

In [ ]:
actuals = train_Y[test_set].values.astype(int)
actuals

In [ ]:
predictions == actuals

In [ ]:
model.predict(train_X[test_set])